In [1]:
import os
import sys
sys.path.append("../src/utils/")
sys.path.append("../src/utils/IEBCS/")
import eventIO
import torch
import numpy as np
import pandas as pd
import shutil

In [2]:
class SimpleNet(torch.nn.Module):

    def __init__(self):
        super(SimpleNet, self).__init__()
        # Define your network layers here
    


    def forward(self, x):
        # Define the forward pass
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        x = torch.relu(x)
        x = self.fc3(x)
        x = torch.relu(x)
        x = self.fc4(x)
        x = torch.tanh(x)
        return x  * 1000

model = torch.load("../data/ball_gun_to_spin.pth", weights_only=False)
print(model)

SimpleNet(
  (fc1): Linear(in_features=2, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=32, bias=True)
  (fc4): Linear(in_features=32, out_features=3, bias=True)
)


In [7]:
files = os.listdir("/home/lkolmar/Documents/metavision/recordings/new_real_dataset/preprocessed/")
files_to_keep = []
for f in files:
    values = f.split("_")
    spin = int(values[0].replace("spin", ""))
    sidespin = int(values[1].replace("sidespin", ""))
    if sidespin == 0:
        if spin in [-5, -4, -3, -2, -1, 6, 2, 4, 5, 7]:
            files_to_keep.append(f)

print(f"Keeping {len(files_to_keep)} files out of {len(files)}")


Keeping 100 files out of 560


In [16]:
idx = 0
for f in files_to_keep:
    values = f.split("_")
    spin = int(values[0].replace("spin", ""))

    spin_vector = model(torch.tensor([[spin, 0.0]]))
    print(f"{f}: {spin_vector.detach().numpy()}")
    rps = np.linalg.norm(spin_vector.detach().numpy()) / (2 * np.pi)
    df = pd.DataFrame({
        "rotation_x": [spin_vector[0, 0].item()],
        "rotation_y": [spin_vector[0, 1].item()],
        "rotation_z": [spin_vector[0, 2].item()],
        "rotation_omega": [rps]
    })
    os.mkdir(f"/home/lkolmar/Documents/metavision/recordings/new_real_dataset/dataset/small_regression/data/{str(idx).zfill(5)}")
    os.mkdir(f"/home/lkolmar/Documents/metavision/recordings/new_real_dataset/dataset/small_regression/preprocessed/{str(idx).zfill(5)}")
    shutil.copyfile(f"/home/lkolmar/Documents/metavision/recordings/new_real_dataset/preprocessed/{f}", 
                    f"/home/lkolmar/Documents/metavision/recordings/new_real_dataset/dataset/small_regression/preprocessed/{str(idx).zfill(5)}/{str(idx).zfill(5)}_roi.hdf5")
    df.to_csv(f"/home/lkolmar/Documents/metavision/recordings/new_real_dataset/dataset/small_regression/data/{str(idx).zfill(5)}/{str(idx).zfill(5)}_ground_truth.csv", index=False)

    idx += 1

spin-4_sidespin0_6.hdf5: [[-17.446283  44.869045 454.96667 ]]
spin4_sidespin0_0.hdf5: [[  27.965054  -25.359943 -297.35837 ]]
spin5_sidespin0_7.hdf5: [[  39.92231  -30.64132 -366.85342]]
spin-3_sidespin0_1.hdf5: [[-27.178432  22.767467 305.02374 ]]
spin7_sidespin0_0.hdf5: [[  43.46403  -72.43989 -812.76245]]
spin6_sidespin0_7.hdf5: [[  42.396008  -44.834232 -566.68195 ]]
spin-3_sidespin0_0.hdf5: [[-27.178432  22.767467 305.02374 ]]
spin-2_sidespin0_1.hdf5: [[-34.768738  16.431871 211.80501 ]]
spin6_sidespin0_8.hdf5: [[  42.396008  -44.834232 -566.68195 ]]
spin7_sidespin0_8.hdf5: [[  43.46403  -72.43989 -812.76245]]
spin-3_sidespin0_4.hdf5: [[-27.178432  22.767467 305.02374 ]]
spin-5_sidespin0_5.hdf5: [[-27.74099   58.932587 600.92377 ]]
spin-3_sidespin0_6.hdf5: [[-27.178432  22.767467 305.02374 ]]
spin6_sidespin0_1.hdf5: [[  42.396008  -44.834232 -566.68195 ]]
spin7_sidespin0_5.hdf5: [[  43.46403  -72.43989 -812.76245]]
spin-3_sidespin0_7.hdf5: [[-27.178432  22.767467 305.02374 ]]
spin